# 02 · 패턴 전처리 (CN7 / RG3)

시안 1~2단계의 입력 검증·동일 입력 패턴 통합·원본 연결을 수행합니다.
`DATASET`을 선택한 뒤 위에서 아래로 실행하세요. 두 데이터는 독립적으로 처리됩니다.
학습·스케일링·라벨 추정은 수행하지 않습니다. 기존 conservative 산출물 경로와 패턴 ID를 유지합니다.


## 1. 작업 경로와 라이브러리 준비

프로젝트 경로를 찾고 전처리에 필요한 라이브러리를 불러옵니다.


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/origin").is_dir()), None)
if ROOT is None:
    raise RuntimeError("26KAMP 프로젝트 폴더에서 노트북을 실행하세요.")
DATASET = "cn7"  # "cn7" 또는 "rg3"
EXPECTED = {
    "cn7": dict(rows=1211, patterns=606, normal=592, risk=14, defects=17, conflicts=11, unlabeled_patterns=28771),
    "rg3": dict(rows=1182, patterns=591, normal=566, risk=25, defects=25, conflicts=25, unlabeled_patterns=29707),
}
if DATASET not in EXPECTED:
    raise ValueError("DATASET은 cn7 또는 rg3이어야 합니다.")
expected = EXPECTED[DATASET]
OUT = ROOT / "data/processed" / DATASET / "conservative"

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


## 2. 원본 데이터 불러오기

라벨·비라벨 원본과 24개 입력 컬럼을 확인하고 원본 파일의 해시를 기록합니다.


In [ ]:
labeled_path = ROOT / f'data/origin/moldset_labeled_{DATASET}.csv'
unlabeled_path = ROOT / f'data/origin/moldset_unlabeled_{DATASET}.csv'
schema_path = ROOT / 'data/schema/input_features.json'
features = json.loads(schema_path.read_text(encoding='utf-8'))['feature_columns']
sources = {str(p.relative_to(ROOT)): digest(p) for p in (labeled_path, unlabeled_path, schema_path)}
labeled = pd.read_csv(labeled_path, float_precision='round_trip')
unlabeled = pd.read_csv(unlabeled_path, float_precision='round_trip')
assert len(features) == 24 and 'PassOrFail' not in features
assert set(labeled['PassOrFail']) == {0, 1}


## 3. 입력 데이터 품질 점검

자료형, 결측값, 무한대, 상수 컬럼, 중복 입력과 원본 ID의 고유성을 확인합니다. 상수 제거는 분할 후 학습 구간에서 수행합니다.


In [ ]:
quality = {}
for name, df in [('labeled', labeled), ('unlabeled', unlabeled)]:
    x = df[features]
    assert all(pd.api.types.is_numeric_dtype(x[c]) for c in features)
    assert np.isfinite(x.to_numpy()).all(), '결측값 또는 무한대가 발견되었습니다. 처리 기준을 먼저 정해야 합니다.'
    quality[name] = {
        'rows': len(df), 'missing_feature_values': int(x.isna().sum().sum()),
        'constant_features': [c for c in features if x[c].nunique(dropna=False) == 1],
        'duplicate_input_rows': int(x.duplicated().sum()),
        'source_id_unique': bool(df['Unnamed: 0'].is_unique),
        'dtypes': {c: str(x[c].dtype) for c in features},
    }



## 4. 불량 우선 패턴 통합과 원본 연결

24개 입력값이 완전히 같은 행을 하나로 통합합니다. 불량이 하나라도 있으면 대표 라벨을 1로 지정하고 원본 행과 관측 횟수를 보존합니다. 비라벨도 동일 입력을 통합하고 모든 원본 행과의 연결을 보존합니다. 비라벨에는 라벨을 생성하지 않습니다.


In [ ]:
# 패턴 ID는 파일별로 구분하며 원본에서 처음 등장한 순서대로 부여합니다.
codes = labeled.groupby(features, sort=False, dropna=False).ngroup()
pattern_ids = codes.map(lambda i: f'{DATASET.upper()}_L_{i:06d}')
work = labeled.assign(pattern_id=pattern_ids, source_row=np.arange(len(labeled)))
groups = work.groupby('pattern_id', sort=False)
metadata = groups.agg(
    normal_count=('PassOrFail', lambda x: int((x == 0).sum())),
    defect_count=('PassOrFail', 'sum'), total_count=('PassOrFail', 'size'),
    representative_label=('PassOrFail', 'max'),
)
metadata['conflicting'] = (metadata.normal_count > 0) & (metadata.defect_count > 0)
metadata['pattern_type'] = np.where(metadata.conflicting, 'conflicting', np.where(metadata.defect_count > 0, 'defect_only', 'normal_only'))
metadata['source_ids'] = groups['Unnamed: 0'].agg(lambda x: json.dumps(x.tolist()))
metadata['source_rows'] = groups['source_row'].agg(lambda x: json.dumps(x.tolist()))
# 위험 패턴은 불량 원본 행을, 정상 패턴은 첫 정상 행을 대표 행으로 선택합니다.
representatives = work.sort_values('PassOrFail', ascending=False, kind='stable').drop_duplicates('pattern_id').set_index('pattern_id').loc[metadata.index]
metadata['representative_source_id'] = representatives['Unnamed: 0']
metadata['representative_source_row'] = representatives['source_row']
X = representatives[features].reset_index(drop=True)
y = metadata[['representative_label']].rename(columns={'representative_label': 'PassOrFail'}).reset_index(drop=True)
metadata = metadata.reset_index()
metadata.insert(0, 'pattern_row', np.arange(len(metadata)))
mapping = work[['source_row', 'Unnamed: 0', 'pattern_id', 'PassOrFail']].rename(columns={'Unnamed: 0': 'source_id', 'PassOrFail': 'original_label'})
mapping = mapping.merge(metadata[['pattern_id', 'pattern_row', 'representative_label']], on='pattern_id', how='left', validate='many_to_one', sort=False)
mapping = mapping.sort_values('source_row').reset_index(drop=True)
# 비라벨도 같은 24개 입력 기준으로 통합하며 첫 원본 행을 대표로 선택합니다.
u_codes = unlabeled.groupby(features, sort=False, dropna=False).ngroup()
u_work = unlabeled.assign(source_row=np.arange(len(unlabeled)),
    pattern_id=u_codes.map(lambda i: f'{DATASET.upper()}_U_{i:06d}'))
u_groups = u_work.groupby('pattern_id', sort=False)
u_meta = u_groups.agg(total_count=('source_row', 'size'),
    representative_source_row=('source_row', 'first'),
    representative_source_id=('Unnamed: 0', 'first'))
u_meta['source_ids'] = u_groups['Unnamed: 0'].agg(lambda x: json.dumps(x.tolist()))
u_meta['source_rows'] = u_groups['source_row'].agg(lambda x: json.dumps(x.tolist()))
u_representatives = u_work.drop_duplicates('pattern_id').set_index('pattern_id').loc[u_meta.index]
U = u_representatives[features].reset_index(drop=True)
u_meta = u_meta.reset_index()
u_meta.insert(0, 'pattern_row', np.arange(len(u_meta)))
u_mapping = u_work[['source_row', 'Unnamed: 0', 'pattern_id']].rename(columns={'Unnamed: 0': 'source_id'})
u_mapping = u_mapping.merge(u_meta[['pattern_id', 'pattern_row']], on='pattern_id',
    how='left', validate='many_to_one', sort=False).sort_values('source_row').reset_index(drop=True)


## 5. 대표 라벨과 원본 복원 검증

패턴 수와 라벨 수, 불량 이력 보존 여부를 확인합니다. 연결표로 복원한 입력이 원본과 정확히 일치하는지도 검증합니다.


In [ ]:
# 모든 원본 행의 연결, 대표 라벨 정책과 입력값의 정확한 복원을 검증합니다.
assert len(X) == expected['patterns'] and y.PassOrFail.value_counts().to_dict() == {0: expected['normal'], 1: expected['risk']}
assert not X.duplicated().any()
assert len(mapping) == len(labeled) and mapping.source_row.is_unique
assert int(metadata.total_count.sum()) == expected['rows']
assert int(metadata.defect_count.sum()) == expected['defects']
assert int(metadata.conflicting.sum()) == expected['conflicts']
assert np.array_equal(X.iloc[mapping.pattern_row.to_numpy()].to_numpy(), labeled[features].to_numpy())
assert (mapping.loc[mapping.original_label == 1, 'representative_label'] == 1).all()
assert np.array_equal(representatives.PassOrFail.to_numpy(), y.PassOrFail.to_numpy())
assert list(X) == list(U) == features


# 통합된 비라벨 입력으로 모든 원본 행을 정확히 복원할 수 있어야 합니다.
assert len(U) == expected['unlabeled_patterns']
assert not U.duplicated().any()
assert len(u_mapping) == len(unlabeled) and u_mapping.source_row.is_unique
assert int(u_meta.total_count.sum()) == len(unlabeled)
assert np.array_equal(U.iloc[u_mapping.pattern_row.to_numpy()].to_numpy(), unlabeled[features].to_numpy())
assert 'PassOrFail' not in U.columns and 'PassOrFail' not in u_meta.columns
# 패턴별 예측도 이 인덱스로 원본 행 순서에 복원할 수 있습니다.
pattern_prediction_example = np.arange(len(U))
restored_prediction_example = pattern_prediction_example[u_mapping.pattern_row.to_numpy()]
assert len(restored_prediction_example) == len(unlabeled)


## 6. 전처리 결과 저장과 재확인

입력, 라벨, 메타데이터와 원본 연결표를 CSV로 저장한 뒤 다시 읽어 값이 유지되는지 확인합니다.


In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
artifacts = {
    'X_labeled.csv': X, 'y_labeled.csv': y,
    'labeled_metadata.csv': metadata, 'source_row_mapping.csv': mapping,
    'X_unlabeled.csv': U, 'unlabeled_metadata.csv': u_meta,
    'unlabeled_source_row_mapping.csv': u_mapping,
}
for filename, frame in artifacts.items():
    frame.to_csv(OUT / filename, index=False, encoding='utf-8')
    restored = pd.read_csv(OUT / filename, float_precision='round_trip')
    pd.testing.assert_frame_equal(restored, frame.reset_index(drop=True), check_dtype=False)


## 7. 전처리 정책과 검증 기록 저장

적용한 정책, 데이터 품질, 파일 해시와 검증 결과를 기록합니다. 분할과 스케일링은 아직 수행하지 않습니다.


In [ ]:
manifest = {
    'dataset': DATASET, 'target': 'PassOrFail', 'label_meaning': {'0': 'no_observed_defect_history', '1': 'observed_defect_history'},
    'feature_columns': features, 'source_sha256': sources,
    'labeled_rows_before': len(labeled), 'labeled_patterns_after': len(X),
    'normal_patterns': expected['normal'], 'risk_patterns': expected['risk'], 'risk_fraction': expected['risk'] / len(X),
    'unlabeled_rows': len(U), 'unlabeled_rows_before': len(unlabeled),
    'unlabeled_unique_patterns': len(U), 'unlabeled_removed_duplicate_rows': len(unlabeled) - len(U),
    'group_policy': 'Exact equality of all 24 inputs; no rounding or approximate grouping; source-local IDs.',
    'label_policy': 'max original label within each labeled pattern; choose a source defect row when label is 1.',
    'unlabeled_policy': 'Aggregate exact 24-feature duplicates; retain first row and reversible source mapping; no labels imputed.',
    'fit_policy': 'No scaling, imputation, constant removal, feature selection or resampling fitted.',
    'scale_alignment': 'Not established between separately standardized labeled and unlabeled sources.',
    'split_status': 'Not performed; planned fixed stratified test 20%, development 80% with stratified 4-fold.',
    'quality': quality,
    'artifact_sha256': {f: digest(OUT / f) for f in artifacts},
}
(OUT / 'preprocessing_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'validation.json').write_text(json.dumps({
    'passed': True, 'checks': [f'{len(X)} unique patterns', f"{expected['normal']} normal and {expected['risk']} risk patterns",
    f'all {len(labeled)} source rows mapped exactly once', f"all {expected['defects']} defect rows retained in risk history",
    f"{expected['conflicts']} conflicting patterns labeled risk", '24 input values reconstructed exactly',
    'unlabeled patterns unique and all source inputs reconstructed exactly',
    'unlabeled prediction mapping verified', 'all CSV round trips equal', 'labeled and unlabeled input schema equal'],
    'source_files_unchanged': all(digest(ROOT / p) == sha for p, sha in sources.items()),
}, indent=2), encoding='utf-8')
(OUT / 'README.md').write_text(f"""# {DATASET.upper()} 패턴 전처리

- 라벨 원본 {len(labeled)}행 → {len(X)}개 패턴 (정상 {expected['normal']}, 위험 {expected['risk']}).
- 비라벨 원본 {len(unlabeled)}행 → {len(U)}개 패턴. 라벨을 생성하지 않습니다.
- X_labeled.csv / y_labeled.csv: 동일 행 순서의 24개 입력과 불량 이력 라벨.
- labeled_metadata.csv / source_row_mapping.csv: 패턴 및 원본 행·ID·관측 라벨 연결.
- X_unlabeled.csv / unlabeled_metadata.csv / unlabeled_source_row_mapping.csv: 비라벨 입력과 원본 연결.
- preprocessing_manifest.json / validation.json: 스키마·정책·품질·해시 검증.

pattern_row와 source_row는 0부터 시작합니다. 상위 폴더의 과거 원본 행 단위 자료와 혼용하지 마세요.
비라벨과 라벨 파일의 표준화 좌표계 정합은 미확인입니다. 모델 변환은 fold train에서만 적합합니다.
다음 단계: preprocessing/03_split.ipynb (고정 Test 20%, 개발 80%의 4-fold).
""", encoding='utf-8')
print(DATASET.upper(), '전처리 저장 및 검증 완료:', OUT)


## 8. 전처리 결과 요약

정상·위험 패턴의 구성과 원본 및 비라벨 행 수를 확인합니다.


In [ ]:
요약 = metadata.groupby(['representative_label', 'pattern_type']).size().reset_index(name='패턴 수')
요약['representative_label'] = 요약['representative_label'].map({0: '정상', 1: '위험'})
요약['pattern_type'] = 요약['pattern_type'].map({'normal_only': '정상만 관측', 'conflicting': '정상·불량 함께 관측', 'defect_only': '불량만 관측'})
요약 = 요약.rename(columns={'representative_label': '대표 라벨', 'pattern_type': '원본 관측 유형'})
print(요약.to_string(index=False))
print('원본 라벨 행 수:', len(mapping))
print('통합된 입력 패턴 수:', len(X))
print('비라벨 원본 행 수:', len(unlabeled))
print('비라벨 통합 패턴 수:', len(U))
print('비라벨 제거 중복 행 수:', len(unlabeled) - len(U))
print('검증 결과: 모든 검사 통과')
